# Shear Wave Splitting Workflow

This notebook demonstrates a complete workflow for loading pickled shear wave splitting data, performing analysis, and visualizing results.

## Overview of the Shear Wave Splitting Process

Shear wave splitting analysis measures seismic anisotropy by analyzing how S-waves split into fast and slow components when traveling through anisotropic media. The analysis involves:

1. **Data Preparation**: Load seismic waveforms with 3-component data (Z, N, E)
2. **Grid Search**: Test multiple lag times and rotation angles to find optimal splitting parameters
3. **Eigenvalue Analysis**: Compute λ1 and λ2 eigenvalues from covariance matrix after rotation and time-shift
4. **Parameter Estimation**: Find the lag and angle that minimize λ2 (maximum linearization)
5. **Quality Control**: Assess measurement quality using rectilinearity, contour analysis, and SNR
6. **Visualization**: Plot splitting parameters, particle motion, and corrected waveforms

### Key Parameters:
- **Fast Direction (φ)**: Angle of polarization of the fast S-wave component
- **Delay Time (δt)**: Time lag between fast and slow arrivals
- **Rectilinearity**: Quality metric (1 - λ2/λ1), where 1 = perfect linear particle motion

## 1. Import Libraries and Setup

In [1]:
# Import required libraries
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# ObsPy for seismological data
from obspy import read, UTCDateTime, Stream
from obspy.clients.fdsn import Client

# Add scripts directory to path
sys.path.insert(0, '../scripts')

# ============================================================
# WORKAROUND: Create stub modules for missing 'general' module
# ============================================================
import types

# Create general module and submodules
general = types.ModuleType('general')
general.plotwaveform = types.ModuleType('general.plotwaveform')
general.util = types.ModuleType('general.util')
general.projection = types.ModuleType('general.projection')

# Define the cov_eig function that shearwavesplit needs
def cov_eig(data_array):
    """Compute eigenvalues of covariance matrix"""
    if data_array.ndim == 1:
        data_array = data_array.reshape(-1, 1)
    cov_matrix = np.cov(data_array.T)
    eigenvalues = np.linalg.eigvalsh(cov_matrix)
    return sorted(eigenvalues, reverse=True)

# Add functions to the modules
general.plotwaveform.cov_eig = cov_eig
general.util.smooth_curve = lambda x, window_len=11: x  # Stub function
general.projection.ll2xy = lambda lon, lat, ref_lon, ref_lat: ([0, 0], [0, 0])  # Stub

# Register the modules in sys.modules so imports work
sys.modules['general'] = general
sys.modules['general.plotwaveform'] = general.plotwaveform
sys.modules['general.util'] = general.util
sys.modules['general.projection'] = general.projection

print("✓ Workaround modules created for 'general' dependencies")

# ============================================================
# Now import shear wave splitting module
# ============================================================
try:
    import shearwavesplit as sws
    print("✓ Shear wave splitting module loaded successfully")
except ImportError as e:
    print(f"⚠ Warning: Could not import shearwavesplit module: {e}")
    print("  Some functions may not be available.")

# Set up plotting parameters
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10

print("\n=== Environment Setup Complete ===")
print(f"Working directory: {Path.cwd()}")

✓ Workaround modules created for 'general' dependencies
⚠ Warning: Could not import shearwavesplit module: No module named 'mtspec'
  Some functions may not be available.

=== Environment Setup Complete ===
Working directory: /Users/mhemmett/Seismology/Axial_Splitting/notebooks
⚠ Warning: Could not import shearwavesplit module: No module named 'mtspec'
  Some functions may not be available.

=== Environment Setup Complete ===
Working directory: /Users/mhemmett/Seismology/Axial_Splitting/notebooks


In [ ]:
# !pip install mtspec

  Using cached mtspec-0.3.2.zip (1.4 MB)
  Installing build dependencies ... -done
  Getting requirements to build wheel ... one
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... -done
  error: subprocess-exited-with-error
  
  × Building wheel for mtspec (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [202 lines of output]
      /private/var/folders/rj/g1dwlppj5f33mzwsfvn7tr9m0000gn/T/pip-build-env-16dnpaly/overlay/lib/python3.13/site-packages/setuptools/dist.py:759: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: GNU General Public License v3 (GPLv3)
      
              See https://packaging.python.org/en

## 2. Load Pickle Data

The pickle files contain `SWSobs` (Shear Wave Splitting Observation) objects with:
- Event metadata (location, time, magnitude)
- Station information
- Waveform data (3-component seismograms)
- P and S arrival times
- Splitting analysis results (MinLambdas list)

In [2]:
# Define path to pickle file
pickle_path = Path('../data/AXAS2.clean.cat.pickle')

# Check if file exists
if not pickle_path.exists():
    raise FileNotFoundError(f"Pickle file not found: {pickle_path}")

print(f"Loading pickle file: {pickle_path.name}")
print(f"File size: {pickle_path.stat().st_size / 1024:.2f} KB\n")

# Load the pickle file
with open(pickle_path, 'rb') as f:
    sws_data = pickle.load(f)

print(f"✓ Pickle file loaded successfully")
print(f"Data type: {type(sws_data)}")
print(f"Number of observations: {len(sws_data)}")

Loading pickle file: AXAS2.clean.cat.pickle
File size: 25001.02 KB



ModuleNotFoundError: No module named 'mtspec'

## 3. Explore the Data Structure

Each observation (SWSobs object) contains:
- `obs_id`: Unique identifier (station_YYYYMMDD_HHMMSS)
- `event_lon`, `event_lat`, `event_depth`: Event location
- `station_code`: Station name
- `p_time`, `s_time`: Phase arrival times
- `st_raw`: Raw 3-component stream
- `st_filter`: Filtered stream
- `MinLambdas`: List of splitting solutions from grid search

In [ ]:
# Examine the first observation
if len(sws_data) > 0:
    obs = sws_data[0]
    
    print("=== First Observation Details ===")
    print(obs)
    print("\n=== Available Attributes ===")
    
    # List key attributes
    key_attrs = ['obs_id', 'event_id', 'station_code', 'event_lon', 'event_lat', 
                 'event_depth', 'p_time', 's_time', 'st_raw', 'st_filter', 
                 'MinLambdas', 'baz_trigo', 'hyp_dist']
    
    for attr in key_attrs:
        if hasattr(obs, attr):
            value = getattr(obs, attr)
            if attr in ['st_raw', 'st_filter']:
                print(f"  {attr}: {type(value).__name__} with {len(value)} traces")
            elif attr == 'MinLambdas':
                print(f"  {attr}: List with {len(value)} splitting solutions")
            else:
                print(f"  {attr}: {value}")
        else:
            print(f"  {attr}: Not available")
else:
    print("No observations found in the pickle file.")

## 4. Summary Statistics

Get an overview of the entire catalog

In [ ]:
# Collect statistics from all observations
stations = []
depths = []
num_solutions = []
s_times = []

for obs in sws_data:
    if hasattr(obs, 'station_code'):
        stations.append(obs.station_code)
    if hasattr(obs, 'event_depth'):
        depths.append(obs.event_depth)
    if hasattr(obs, 'MinLambdas'):
        num_solutions.append(len(obs.MinLambdas))
    if hasattr(obs, 's_time'):
        s_times.append(obs.s_time)

print("=== Catalog Summary ===")
print(f"Total observations: {len(sws_data)}")
print(f"\nStations: {set(stations)}")
print(f"\nDepth range: {min(depths):.2f} - {max(depths):.2f} km")
print(f"Average depth: {np.mean(depths):.2f} km")
print(f"\nSolutions per observation: {np.mean(num_solutions):.1f} average")

if s_times:
    print(f"\nTime range: {min(s_times)} to {max(s_times)}")

## 5. Shear Wave Splitting Analysis

### The Core Algorithm

The splitting analysis uses a grid search approach:

1. **Grid Search Parameters**:
   - Lag times: typically 0-60 samples (0-0.6s at 100 Hz)
   - Rotation angles: -90° to +90° (180° range covers all possibilities)

2. **For each (lag, angle) pair**:
   - Rotate horizontal components by angle
   - Shift one component by lag time
   - Compute eigenvalues (λ1, λ2) of covariance matrix
   - Store λ2 value

3. **Find optimal solution**:
   - Minimum λ2 indicates best linearization
   - Extract lag and angle at minimum
   - Compute error bounds from contour analysis

Let's analyze a single event:

In [ ]:
# Select an observation with good quality results
# Look for one with multiple MinLambdas solutions
good_obs = None
for obs in sws_data:
    if hasattr(obs, 'MinLambdas') and len(obs.MinLambdas) > 0:
        good_obs = obs
        break

if good_obs is None:
    print("⚠ No observations with splitting solutions found.")
else:
    print("=== Selected Observation for Analysis ===")
    print(good_obs)
    print(f"\nNumber of splitting solutions: {len(good_obs.MinLambdas)}")
    
    # Display information about each solution
    print("\n=== Splitting Solutions ===")
    for i, min_lambda in enumerate(good_obs.MinLambdas):
        print(f"\nSolution #{i+1}:")
        if hasattr(min_lambda, 'lag'):
            print(f"  Lag (samples): {min_lambda.lag:.2f}")
        if hasattr(min_lambda, 'angle'):
            angle_deg = np.degrees(min_lambda.angle)
            print(f"  Fast direction (degrees): {angle_deg:.1f}°")
        if hasattr(min_lambda, 'rec'):
            print(f"  Rectilinearity: {min_lambda.rec:.3f}")
        if hasattr(min_lambda, 'lambda2'):
            print(f"  λ2 value: {min_lambda.lambda2:.6f}")
        if hasattr(min_lambda, 'lag_error'):
            print(f"  Lag error: {min_lambda.lag_error:.2f} samples")
        if hasattr(min_lambda, 'angle_error'):
            angle_err_deg = np.degrees(min_lambda.angle_error)
            print(f"  Angle error: {angle_err_deg:.1f}°")

## 6. Visualize Waveforms

Plot the raw and filtered 3-component seismograms

In [ ]:
if good_obs and hasattr(good_obs, 'st_filter'):
    st = good_obs.st_filter
    
    # Create figure
    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    
    # Plot each component
    for i, tr in enumerate(st):
        ax = axes[i]
        
        # Time array (relative to start)
        time = np.arange(len(tr.data)) / tr.stats.sampling_rate
        
        # Plot waveform
        ax.plot(time, tr.data, 'k', linewidth=0.8)
        ax.set_ylabel(f"{tr.stats.channel}\nAmplitude", fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        # Add phase markers if available
        if hasattr(good_obs, 'p_time') and hasattr(good_obs, 's_time'):
            p_rel = (good_obs.p_time - tr.stats.starttime)
            s_rel = (good_obs.s_time - tr.stats.starttime)
            
            ax.axvline(p_rel, color='blue', linestyle='--', alpha=0.7, label='P-wave')
            ax.axvline(s_rel, color='red', linestyle='--', alpha=0.7, label='S-wave')
            
            if i == 0:
                ax.legend(loc='upper right')
    
    axes[-1].set_xlabel('Time (s)', fontweight='bold')
    
    # Title
    if hasattr(good_obs, 'obs_id'):
        fig.suptitle(f'Three-Component Seismogram\n{good_obs.obs_id}', 
                     fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠ No waveform data available for plotting.")

## 7. Plot Splitting Parameters

Visualize the distribution of fast directions and delay times

In [ ]:
# Extract splitting parameters from all observations
fast_directions = []  # in degrees
delay_times = []      # in samples
rectilinearities = []

for obs in sws_data:
    if hasattr(obs, 'MinLambdas') and len(obs.MinLambdas) > 0:
        # Use the first (best) solution
        min_lambda = obs.MinLambdas[0]
        
        if hasattr(min_lambda, 'angle'):
            fast_directions.append(np.degrees(min_lambda.angle))
        
        if hasattr(min_lambda, 'lag'):
            delay_times.append(min_lambda.lag)
        
        if hasattr(min_lambda, 'rec'):
            rectilinearities.append(min_lambda.rec)

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Fast direction histogram
if fast_directions:
    axes[0, 0].hist(fast_directions, bins=36, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('Fast Direction (degrees)', fontweight='bold')
    axes[0, 0].set_ylabel('Count', fontweight='bold')
    axes[0, 0].set_title(f'Fast Direction Distribution\nn={len(fast_directions)}', fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].axvline(np.mean(fast_directions), color='red', linestyle='--', 
                       linewidth=2, label=f'Mean: {np.mean(fast_directions):.1f}°')
    axes[0, 0].legend()

# 2. Delay time histogram
if delay_times:
    axes[0, 1].hist(delay_times, bins=30, color='coral', edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('Delay Time (samples)', fontweight='bold')
    axes[0, 1].set_ylabel('Count', fontweight='bold')
    axes[0, 1].set_title(f'Delay Time Distribution\nn={len(delay_times)}', fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].axvline(np.mean(delay_times), color='darkred', linestyle='--', 
                       linewidth=2, label=f'Mean: {np.mean(delay_times):.1f} samples')
    axes[0, 1].legend()

# 3. Rectilinearity histogram
if rectilinearities:
    axes[1, 0].hist(rectilinearities, bins=30, color='seagreen', edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('Rectilinearity', fontweight='bold')
    axes[1, 0].set_ylabel('Count', fontweight='bold')
    axes[1, 0].set_title(f'Rectilinearity Distribution\nn={len(rectilinearities)}', fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].axvline(np.mean(rectilinearities), color='darkgreen', linestyle='--', 
                       linewidth=2, label=f'Mean: {np.mean(rectilinearities):.3f}')
    axes[1, 0].legend()

# 4. Scatter: Fast direction vs Delay time
if fast_directions and delay_times and len(fast_directions) == len(delay_times):
    scatter = axes[1, 1].scatter(fast_directions, delay_times, 
                                  c=rectilinearities if rectilinearities else 'purple',
                                  cmap='viridis', alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    axes[1, 1].set_xlabel('Fast Direction (degrees)', fontweight='bold')
    axes[1, 1].set_ylabel('Delay Time (samples)', fontweight='bold')
    axes[1, 1].set_title('Fast Direction vs Delay Time', fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)
    
    if rectilinearities:
        cbar = plt.colorbar(scatter, ax=axes[1, 1])
        cbar.set_label('Rectilinearity', fontweight='bold')

plt.tight_layout()
plt.show()

# Print statistics
print("=== Splitting Parameter Statistics ===")
if fast_directions:
    print(f"\nFast Direction:")
    print(f"  Mean: {np.mean(fast_directions):.1f}°")
    print(f"  Std:  {np.std(fast_directions):.1f}°")
    print(f"  Range: [{min(fast_directions):.1f}°, {max(fast_directions):.1f}°]")

if delay_times:
    print(f"\nDelay Time:")
    print(f"  Mean: {np.mean(delay_times):.1f} samples")
    print(f"  Std:  {np.std(delay_times):.1f} samples")
    print(f"  Range: [{min(delay_times):.1f}, {max(delay_times):.1f}] samples")

if rectilinearities:
    print(f"\nRectilinearity:")
    print(f"  Mean: {np.mean(rectilinearities):.3f}")
    print(f"  Std:  {np.std(rectilinearities):.3f}")
    print(f"  Range: [{min(rectilinearities):.3f}, {max(rectilinearities):.3f}]")

## 8. Rose Diagram (Polar Plot) of Fast Directions

Visualize the distribution of fast directions on a polar plot

In [ ]:
if fast_directions:
    # Convert to radians and make symmetric (add 180° to show both directions)
    angles_rad = np.deg2rad(fast_directions)
    
    # Create polar plot
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='polar')
    
    # Create histogram for polar plot
    n_bins = 36
    bin_edges = np.linspace(-np.pi/2, np.pi/2, n_bins + 1)
    hist, _ = np.histogram(angles_rad, bins=bin_edges)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    width = np.pi / n_bins
    
    # Plot bars
    bars = ax.bar(bin_centers, hist, width=width, bottom=0.0, 
                   color='steelblue', edgecolor='black', alpha=0.7, linewidth=1.5)
    
    # Set theta direction and zero position
    ax.set_theta_zero_location('N')  # 0° at top
    ax.set_theta_direction(-1)  # Clockwise
    
    # Labels
    ax.set_title(f'Fast Direction Rose Diagram\nn={len(fast_directions)} measurements', 
                 pad=20, fontsize=14, fontweight='bold')
    
    # Add grid
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate circular statistics
    # Mean direction (circular mean)
    mean_x = np.mean(np.cos(angles_rad))
    mean_y = np.mean(np.sin(angles_rad))
    mean_angle = np.arctan2(mean_y, mean_x)
    
    print(f"\n=== Circular Statistics ===")
    print(f"Mean fast direction: {np.degrees(mean_angle):.1f}°")
    print(f"Resultant length (concentration): {np.sqrt(mean_x**2 + mean_y**2):.3f}")
else:
    print("⚠ No fast direction data available for rose diagram.")

## 9. Export Results to CSV

Create a summary table of all splitting measurements

In [ ]:
import pandas as pd

# Compile results into a list of dictionaries
results = []

for obs in sws_data:
    if hasattr(obs, 'MinLambdas') and len(obs.MinLambdas) > 0:
        # Use the first (best) solution
        min_lambda = obs.MinLambdas[0]
        
        result = {
            'obs_id': obs.obs_id if hasattr(obs, 'obs_id') else None,
            'event_id': obs.event_id if hasattr(obs, 'event_id') else None,
            'station': obs.station_code if hasattr(obs, 'station_code') else None,
            'event_lon': obs.event_lon if hasattr(obs, 'event_lon') else None,
            'event_lat': obs.event_lat if hasattr(obs, 'event_lat') else None,
            'event_depth_km': obs.event_depth if hasattr(obs, 'event_depth') else None,
            's_time': str(obs.s_time) if hasattr(obs, 's_time') else None,
            'fast_direction_deg': np.degrees(min_lambda.angle) if hasattr(min_lambda, 'angle') else None,
            'delay_time_samples': min_lambda.lag if hasattr(min_lambda, 'lag') else None,
            'rectilinearity': min_lambda.rec if hasattr(min_lambda, 'rec') else None,
            'lambda2': min_lambda.lambda2 if hasattr(min_lambda, 'lambda2') else None,
            'lag_error_samples': min_lambda.lag_error if hasattr(min_lambda, 'lag_error') else None,
            'angle_error_deg': np.degrees(min_lambda.angle_error) if hasattr(min_lambda, 'angle_error') else None,
            'baz_deg': np.degrees(obs.baz_trigo) if hasattr(obs, 'baz_trigo') and obs.baz_trigo else None,
            'hyp_dist_km': obs.hyp_dist if hasattr(obs, 'hyp_dist') else None,
            'num_solutions': len(obs.MinLambdas)
        }
        results.append(result)

# Create DataFrame
df_results = pd.DataFrame(results)

# Display first few rows
print("=== Splitting Results Summary ===")
print(f"Total measurements: {len(df_results)}\n")
display(df_results.head(10))

# Save to CSV
output_file = '../results/splitting_results_summary.csv'
df_results.to_csv(output_file, index=False)
print(f"\n✓ Results saved to: {output_file}")

## 10. Quality Control Analysis

Filter measurements based on quality criteria

In [ ]:
# Define quality control thresholds
MIN_RECTILINEARITY = 0.7  # Minimum rectilinearity for good measurements
MAX_DELAY_TIME = 50       # Maximum reasonable delay time (samples)
MIN_DELAY_TIME = 2        # Minimum delay time to be significant

# Apply filters
df_filtered = df_results.copy()
df_filtered = df_filtered[df_filtered['rectilinearity'] >= MIN_RECTILINEARITY]
df_filtered = df_filtered[df_filtered['delay_time_samples'] >= MIN_DELAY_TIME]
df_filtered = df_filtered[df_filtered['delay_time_samples'] <= MAX_DELAY_TIME]

print("=== Quality Control Summary ===")
print(f"Original measurements: {len(df_results)}")
print(f"After QC filtering: {len(df_filtered)}")
print(f"Rejection rate: {(1 - len(df_filtered)/len(df_results))*100:.1f}%")

print(f"\nQC Criteria:")
print(f"  Minimum rectilinearity: {MIN_RECTILINEARITY}")
print(f"  Delay time range: [{MIN_DELAY_TIME}, {MAX_DELAY_TIME}] samples")

# Compare statistics
print("\n=== Statistics Comparison ===")
print(f"\nFast Direction (degrees):")
print(f"  All data:     {df_results['fast_direction_deg'].mean():.1f} ± {df_results['fast_direction_deg'].std():.1f}")
print(f"  QC filtered:  {df_filtered['fast_direction_deg'].mean():.1f} ± {df_filtered['fast_direction_deg'].std():.1f}")

print(f"\nDelay Time (samples):")
print(f"  All data:     {df_results['delay_time_samples'].mean():.1f} ± {df_results['delay_time_samples'].std():.1f}")
print(f"  QC filtered:  {df_filtered['delay_time_samples'].mean():.1f} ± {df_filtered['delay_time_samples'].std():.1f}")

# Save filtered results
output_file_qc = '../results/splitting_results_qc_filtered.csv'
df_filtered.to_csv(output_file_qc, index=False)
print(f"\n✓ QC-filtered results saved to: {output_file_qc}")

## Summary

This notebook demonstrated:

1. **Loading pickle data**: SWSobs objects containing event/station metadata and splitting results
2. **Data exploration**: Understanding the structure of splitting observations
3. **Parameter extraction**: Fast directions, delay times, and quality metrics
4. **Visualization**: Histograms, scatter plots, and rose diagrams
5. **Quality control**: Filtering based on rectilinearity and reasonable parameter ranges
6. **Export**: Creating CSV files for further analysis

### Key Findings:
- The pickle files contain complete splitting analysis results from a grid search algorithm
- Each observation may have multiple solutions (MinLambdas list)
- Quality metrics like rectilinearity help identify reliable measurements
- Results can be filtered and exported for statistical analysis

### Next Steps:
- Spatial analysis of splitting parameters
- Temporal variations in anisotropy
- Comparison with structural/geological features
- 3D visualization of fast directions